# Ingest CMO emission observations from a `.txt` log into Neo4j

This notebook parses CMO Lua `PY_CONTACT_LOG` lines from a text file into the repository's `EmissionObservation` ontology, writes a reviewable JSONL artifact, and ingests the observations into the same Neo4j evidence graph used by `ingest-graph`.

Use it after running the CMO Lua exporter that emits lines such as:

```text
PY_CONTACT_LOG Time : 123, Sensor_aircraft : Eagle #1, Emission_sensor_name : N019 Radar, Emission_age : 0, Emission_solid : true, Emission_latitude : 12.3, Emission_longitude : 45.6, Emission_target_type : Fighter
```

The notebook also includes an optional static-reference ingestion section that reuses `combat_id_calibration.graph_ingest` for PDFs or Wikipedia pages, so static facts and dynamic CMO observations live in one graph.


## 1. Configure paths and services

Install graph extras in the active kernel environment if needed:

```bash
python -m pip install -e ..[graph]
```

Make sure Neo4j is reachable over Bolt before running ingestion cells. The Ollama settings are only needed for the optional `ingest-graph` static-reference section.


In [ ]:
from pathlib import Path
import os

# Update this to your CMO Lua history/export text file.
CMO_OBSERVATIONS_TXT = Path("../LuaHistory.txt")

# Review artifact written before Neo4j ingestion.
OBSERVATIONS_JSONL = Path("../observations.jsonl")

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE") or None

# Optional graph-ingest settings for static references.
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen3.5:9b")
FACTS_JSONL = Path("../facts.jsonl")


## 2. Import the observation and graph ingestion helpers

`EmissionObservation`, `parse_observations`, `write_observations_jsonl`, and `populate_observations_neo4j` implement the CMO observation route. The optional `graph_ingest` imports are used later for reference PDFs or Wikipedia pages.


In [ ]:
from dataclasses import asdict
import json

from combat_id_calibration.cmo_observation_ingest import (
    EmissionObservation,
    parse_observations,
    populate_observations_neo4j,
    write_observations_jsonl,
)
from combat_id_calibration.graph_ingest import (
    extract_facts,
    load_documents,
    populate_neo4j,
    write_facts_jsonl,
)


## 3. Parse `PY_CONTACT_LOG` lines from the `.txt` file

Non-observation lines are ignored. Each parsed row is a typed `EmissionObservation` with normalized booleans and numeric fields where possible.


In [ ]:
if not CMO_OBSERVATIONS_TXT.exists():
    raise FileNotFoundError(
        f"Set CMO_OBSERVATIONS_TXT to an existing .txt log file; current value: {CMO_OBSERVATIONS_TXT}"
    )

with CMO_OBSERVATIONS_TXT.open("r", encoding="utf-8-sig", errors="replace") as handle:
    observations: list[EmissionObservation] = list(parse_observations(handle))

print(f"Parsed {len(observations)} observation(s) from {CMO_OBSERVATIONS_TXT}")
if observations:
    print(json.dumps(asdict(observations[0]), indent=2, sort_keys=True))


## 4. Write a reviewable observations JSONL artifact

Review this file before ingestion. It is deterministic and can be archived with the run for provenance.


In [ ]:
write_observations_jsonl(observations, OBSERVATIONS_JSONL)
print(f"Wrote {len(observations)} observation(s) to {OBSERVATIONS_JSONL}")


## 5. Ingest observations into Neo4j

This creates/updates `Observation`, `Contact`, `Sensor`, `Emission`, `Platform`, `PlatformClass`, and `Source` nodes and links them with the observation ontology relationships.


In [ ]:
if not NEO4J_PASSWORD:
    raise ValueError("Set NEO4J_PASSWORD in the environment before running Neo4j ingestion.")

populate_observations_neo4j(
    observations,
    uri=NEO4J_URI,
    user=NEO4J_USER,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
)
print(f"Ingested {len(observations)} CMO observation(s) into {NEO4J_URI}")


## 6. Equivalent CLI command

Use this from a shell when you do not need notebook inspection:


In [ ]:
print(
    "python -m combat_id_calibration ingest-cmo-observations "
    f"--input {CMO_OBSERVATIONS_TXT} "
    f"--observations-jsonl {OBSERVATIONS_JSONL} "
    f"--neo4j-uri {NEO4J_URI} "
    f"--neo4j-user {NEO4J_USER} "
    '--neo4j-password "$NEO4J_PASSWORD"'
    + (f" --neo4j-database {NEO4J_DATABASE}" if NEO4J_DATABASE else "")
)


## 7. Optional: ingest static reference facts with `ingest-graph`

Run this section when you also want reference facts from PDFs or Wikipedia pages in the same Neo4j database. The extraction step requires Ollama and the configured local model.


In [ ]:
PDF_PATHS = ["../34_A_Holistic_Approach_to_Combat_Identification_200701.pdf"]
WIKIPEDIA_URLS = [
    # "https://en.wikipedia.org/wiki/Mikoyan_MiG-29",
    # "https://en.wikipedia.org/wiki/N019_radar",
]

# Uncomment to run static graph ingestion.
# documents = load_documents(PDF_PATHS, WIKIPEDIA_URLS)
# facts = extract_facts(
#     documents,
#     model=OLLAMA_MODEL,
#     ollama_url=OLLAMA_URL,
#     max_chars=6000,
#     overlap=500,
#     diagnostics=True,
# )
# write_facts_jsonl(facts, FACTS_JSONL)
# populate_neo4j(facts, NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE)
# print(f"Ingested {len(facts)} static reference fact(s) into {NEO4J_URI}")


## 8. Optional Cypher sanity checks

Run these in Neo4j Browser or another Cypher client after ingestion:

```cypher
MATCH (o:Observation) RETURN count(o) AS observations;
MATCH (c:Contact)-[:HAS_OBSERVATION]->(o:Observation)-[:OBSERVED_BY]->(p:Platform)
RETURN c.name, o.time, p.name, o.latitude, o.longitude
ORDER BY o.time
LIMIT 25;
MATCH (contact:Contact)-[:CLASSIFIED_AS]->(class:PlatformClass)
RETURN class.name, count(contact) AS contacts
ORDER BY contacts DESC;
```
